# FIUBench GA-forget5 학습 — Fast tokenizer, Kaggle T4 ×2

이 노트북은 처음부터 끝까지 같은 고정 환경을 사용합니다.

- `transformers==4.46.3`
- `tokenizers==0.20.3`
- `LlamaTokenizerFast` (`use_fast=True`)
- `peft==0.13.2`, `accelerate==1.14.0`, `deepspeed==0.19.7`
- GA Forget5, LoRA 128/256/0.05, LR 2e-5, 8 epochs
- GPU당 batch 4 × GPU 2 × accumulation 16 = effective forget batch 128

GA loss는 retain batch를 사용하지 않으며 `data_module.py`는 수정하지 않습니다. 실제 400개 학습
문장의 tokenizer class, input IDs, labels를 학습 전에 검사하고 그 signature를 저장합니다.


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working
if [ ! -d FIUBench_kaggle ]; then
  git clone https://github.com/npngjjm/FIUBench_kaggle.git
fi
python -m pip uninstall -y torchao || true
python -m pip install --no-cache-dir --upgrade \
  'transformers==4.46.3' \
  'tokenizers==0.20.3' \
  'peft==0.13.2' \
  'accelerate==1.14.0' \
  'deepspeed==0.19.7' \
  'huggingface_hub==0.36.2' \
  'numpy==2.1.3' \
  hydra-core omegaconf datasets sentencepiece protobuf scipy pillow \
  pyyaml requests tqdm rouge_score scikit-learn

echo 'INSTALL_COMPLETE — 커널을 한 번 재시작하세요.'


설치 후 아래 셀을 한 번 실행해 Python 커널을 재시작하세요. 연결이 돌아오면 재시작 셀은
다시 실행하지 말고 환경 확인 셀부터 진행합니다.


In [ ]:
import os
os._exit(0)


In [ ]:
import os
from pathlib import Path
from importlib.metadata import version
import torch

repo = Path('/kaggle/working/FIUBench_kaggle')
assert repo.is_dir(), '저장소가 없습니다. clone/설치 셀을 다시 실행하세요.'
os.chdir(repo)
os.environ['HF_HOME'] = '/kaggle/working/hf_cache'

expected = {
    'transformers': '4.46.3',
    'tokenizers': '0.20.3',
    'peft': '0.13.2',
    'accelerate': '1.14.0',
    'deepspeed': '0.19.7',
    'huggingface-hub': '0.36.2',
    'numpy': '2.1.3',
}
for package, wanted in expected.items():
    installed = version(package)
    print(package, installed)
    assert installed == wanted, (package, installed, wanted)

print('torch:', torch.__version__)
print('GPU count:', torch.cuda.device_count())
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() == 2


In [ ]:
# 400명 전원의 이미지를 공개 FIUBench dataset revision에서 복원합니다.
import json, shutil, zipfile
from collections import defaultdict
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download
from PIL import Image
repo=Path('/kaggle/working/FIUBench_kaggle')
records=[json.loads(s) for s in (repo/'dataset/full.json').read_text().splitlines() if s.strip()][:400]
needed={r['image_path'] for r in records}
assert len(needed)==400
rev=HfApi().dataset_info('gray311/FIUBench').sha
archive=hf_hub_download('gray311/FIUBench','SFHQ.zip',repo_type='dataset',revision=rev,local_dir='/kaggle/working/fiu_download')
with zipfile.ZipFile(archive) as z:
    entries=defaultdict(list)
    for m in z.infolist():
        p=Path(m.filename)
        if not m.is_dir() and '__MACOSX' not in p.parts and not p.name.startswith('._'):
            entries[p.name].append(m)
    for image_path in needed:
        dest=(repo/image_path).resolve()
        assert dest.is_relative_to((repo/'dataset').resolve())
        hits=entries[dest.name]; assert len(hits)==1,(dest.name,len(hits))
        dest.parent.mkdir(parents=True,exist_ok=True)
        with z.open(hits[0]) as src,dest.open('wb') as dst: shutil.copyfileobj(src,dst)
        with Image.open(dest) as im: im.verify()
assert all((repo/p).is_file() for p in needed)
Path('/kaggle/working/fiu_ga_data_revision.txt').write_text(rev)
print('이미지 검증:',len(needed),'dataset revision:',rev)


In [ ]:
# Kaggle 런타임 forget.py에 메모리 호환 패치와 fast tokenizer를 명시합니다.
# GA loss와 data_module.py는 변경하지 않습니다.
import ast
from pathlib import Path

path = repo / "forget.py"
text = path.read_text()
original = text

old = "AutoTokenizer.from_pretrained(cfg.model_path)"
new = "AutoTokenizer.from_pretrained(cfg.model_path, use_fast=True)"
assert old in text or new in text
text = text.replace(old, new)

marker = "    if cfg.LoRA.r != 0:\n        config = LoraConfig("
block = """    if str(model_cfg.get("gradient_checkpointing", "false")).lower() == "true":
        model.config.use_cache = False
        model.gradient_checkpointing_enable()
        model.enable_input_require_grads()
        logger.info("Gradient checkpointing enabled", main_process_only=True)

"""
if "model.gradient_checkpointing_enable()" not in text:
    assert text.count(marker) == 1
    text = text.replace(marker, block + marker, 1)

end = text.rfind("\nif __name__ == \"__main__\":")
assert end > 0
if "if cfg.save_model and accelerator.is_main_process:" not in text:
    start = text.rfind("\n    accelerator.end_training()", 0, end)
    assert start > 0
    replacement = """
    output_dir = cfg.save_dir
    accelerator.wait_for_everyone()
    if cfg.save_model and accelerator.is_main_process:
        os.makedirs(output_dir, exist_ok=True)
        unwrapped_model = accelerator.unwrap_model(model)
        if cfg.LoRA.r != 0:
            save_lora_weights(unwrapped_model, output_dir)
        else:
            unwrapped_model.save_pretrained(output_dir)
            tokenizer.save_pretrained(output_dir)
    accelerator.wait_for_everyone()
    accelerator.end_training()
"""
    text = text[:start] + replacement + text[end:]

ast.parse(text)
backup = path.with_suffix(".py.before_fast_kaggle_patch")
if not backup.exists():
    backup.write_text(original)
path.write_text(text)
assert "use_fast=True" in text
assert "use_fast=False" not in text
assert "model.enable_input_require_grads()" in text
print("FAST tokenizer + Kaggle memory/save patch complete")


In [ ]:
# GA Forget5 config와 T4×2 Accelerate config를 명시적으로 작성합니다.
(repo / 'config/forget_ga_forget5_fast.yaml').write_text("""defaults:
  - forget_lora
  - _self_
model_family: llava-phi
model_path: gray311/vlm_unlearning_ft_llava_phi_3_mini
attn_implementation: sdpa
data_path: ./dataset/full.json
split: forget5
forget_loss: ga
lr: 2e-5
batch_size: 4
gradient_accumulation_steps: 16
num_epochs: 8
LoRA: {r: 128, alpha: 256, dropout: 0.05}
workers: 0
report_to: null
save_dir: /kaggle/working/models/llava_phi_GA_forget5_fast
save_model: true
save_steps: 4
""")
(repo / 'config/accelerate_kaggle_fast_t4x2.yaml').write_text("""compute_environment: LOCAL_MACHINE
debug: false
deepspeed_config:
  gradient_accumulation_steps: 16
  gradient_clipping: 1.0
  offload_optimizer_device: cpu
  offload_param_device: cpu
  zero3_init_flag: false
  zero_stage: 2
distributed_type: DEEPSPEED
downcast_bf16: 'no'
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 2
rdzv_backend: static
same_network: true
use_cpu: false
""")
print((repo / 'config/forget_ga_forget5_fast.yaml').read_text())


In [ ]:
# 실제 GA 학습 400문장의 fast-tokenizer input_ids와 labels를 전부 검증합니다.
import hashlib
import json
import sys
from transformers import AutoTokenizer

if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
from data_module import preprocess_v1
from utils import get_model_identifiers_from_yaml

model_path = 'gray311/vlm_unlearning_ft_llava_phi_3_mini'
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)
assert tokenizer.is_fast
assert tokenizer.__class__.__name__ == 'LlamaTokenizerFast'

model_cfg = get_model_identifiers_from_yaml('llava-phi')
question_tag = model_cfg['question_start_tag']
answer_tag = model_cfg['answer_tag']
roles = [question_tag, answer_tag]

split = json.loads((repo / 'dataset/split.json').read_text())
rows = [json.loads(x) for x in (repo / 'dataset/full.json').read_text().splitlines() if x.strip()][:400]
forget_ids = set(split['forget5'])
samples = [qa for row in rows if row['unique_id'] in forget_ids for qa in row['qa_list']]
assert len(forget_ids) == 20 and len(samples) == 400

digest = hashlib.sha256()
assistant_id = tokenizer.convert_tokens_to_ids('<|assistant|>')
for qa in samples:
    question = qa['question'].capitalize()
    answer = qa['answer'].capitalize()
    conversation = question_tag + '<image>\n' + question + answer_tag + answer
    encoded = tokenizer(conversation, max_length=64, truncation=True, return_tensors='pt')
    labels = preprocess_v1(tokenizer, encoded['input_ids'], conversation, roles)
    supervised = labels[0][labels[0] != -100]
    assert supervised.numel() > 1
    assert int(supervised[0]) == assistant_id
    digest.update(encoded['input_ids'].numpy().tobytes())
    digest.update(labels.numpy().tobytes())

signature = digest.hexdigest()
(repo / 'ga_fast_training_token_signature.txt').write_text(signature + '\n')
print('tokenizer:', tokenizer.__class__.__name__, 'is_fast:', tokenizer.is_fast)
print('forget identities:', len(forget_ids), 'forget QA:', len(samples))
print('assistant token id:', assistant_id)
print('400-sample input/label SHA256:', signature)
print('effective forget batch:', 4 * 2 * 16)
print('optimizer steps:', 32)
print('FAST_GA_TRAINING_PREFLIGHT_OK')


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/FIUBench_kaggle
export HF_HOME=/kaggle/working/hf_cache
export TOKENIZERS_PARALLELISM=false
export HYDRA_FULL_ERROR=1
export PYTHONUNBUFFERED=1
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
CUDA_VISIBLE_DEVICES=0,1 DS_SKIP_CUDA_CHECK=1 accelerate launch \
  --config_file config/accelerate_kaggle_fast_t4x2.yaml \
  --main_process_port 2233 \
  forget.py --config-name forget_ga_forget5_fast \
  2>&1 | tee /kaggle/working/ga_forget5_fast_console.log
echo TRAIN_PROCESS_EXIT_0


In [ ]:
from pathlib import Path
import torch

out = Path('/kaggle/working/models/llava_phi_GA_forget5_fast')
checkpoints = sorted(out.glob('step_*/checkpoint.pt'))
print('중간 checkpoints:', [str(p.parent) for p in checkpoints])
assert len(checkpoints) == 8
for checkpoint_path in checkpoints:
    print(checkpoint_path.parent.name, round(checkpoint_path.stat().st_size / 1024**2, 1), 'MiB')

final_checkpoint = out / 'checkpoint.pt'
assert final_checkpoint.is_file(), final_checkpoint
state = torch.load(final_checkpoint, map_location='cpu', weights_only=True)
assert isinstance(state, dict) and state
print('최종 checkpoint:', final_checkpoint)
print('최종 크기:', round(final_checkpoint.stat().st_size / 1024**2, 1), 'MiB')
print('저장 tensor 수:', len(state))
print('GA FAST checkpoint 전체 검증 통과')


학습이 끝나면 `/kaggle/working/models/llava_phi_GA_forget5_fast`,
`ga_forget5_fast_console.log`, `ga_fast_training_token_signature.txt`,
`fiu_ga_data_revision.txt`를 **Save Version → Quick Save**로 보존하세요.

평가는 `FIUBench_forget5_fast_eval_kaggle.ipynb`에서 `METHOD = "GA"`로 실행합니다.
